In [1]:
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

from transformers_sae import _autoreload

# from transformers_sae import _autoreload
from transformers_sae.ops import MemoryTrackingMode
from transformers_sae.replacement_model import GemmaReplacement, make_replacement_model

# Tweak TRAINING_BATCH_SIZE for your hardware if necessary
if torch.cuda.is_available():
    TRAINING_DEVICE = "cuda:0"
    TRAINING_BATCH_SIZE = 2
elif torch.mps.is_available():
    TRAINING_DEVICE = "mps:0"
    TRAINING_BATCH_SIZE = 2
else:
    TRAINING_DEVICE = "cpu"
    TRAINING_BATCH_SIZE = 2

model_id = "google/gemma-2-2b"
tokenizer = AutoTokenizer.from_pretrained(model_id)
training_dataset = load_dataset(
    "monology/pile-uncopyrighted-parquet",
    split="train",
    streaming=True,
    columns=["text"],
)
validation_dataset = load_dataset(
    "monology/pile-test-val",
    split="validation",
    revision="refs/convert/parquet",
    streaming=True,
    columns=["text"],
)

with MemoryTrackingMode() as mtm:
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        device_map=TRAINING_DEVICE,
        dtype=torch.bfloat16,
        use_safetensors=True,
    )
    model = make_replacement_model(
        model,
        {},
        num_layers=model.config.num_hidden_layers,
        context_length=1024,  # model.config.max_position_embeddings,
        d_model=model.config.hidden_size,
        layer_path="model.layers",
        replacement_class=GemmaReplacement,
    )
    model.eval()
    model.requires_grad_(False)

print(model)
print(mtm.memory_max)
print(mtm.memory_cur)

/Users/evanlloyd/mechinterp-experiments/transformers_sae/.venv/lib/python3.13/site-packages/codefind/registry.py:46: FutureWarning: `torch.distributed.reduce_op` is deprecated, please use `torch.distributed.ReduceOp` instead
  if isinstance(obj, types.FunctionType):


Resolving data files:   0%|          | 0/1987 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1987 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

GemmaReplacementInstance(
  (model): Gemma2Model(
    (embed_tokens): Embedding(256000, 2304, padding_idx=0)
    (layers): ModuleList(
      (0-25): 26 x Gemma2DecoderLayer(
        (self_attn): Gemma2Attention(
          (q_proj): Linear(in_features=2304, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2304, out_features=1024, bias=False)
          (v_proj): Linear(in_features=2304, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2304, bias=False)
        )
        (mlp): Gemma2MLP(
          (gate_proj): Linear(in_features=2304, out_features=9216, bias=False)
          (up_proj): Linear(in_features=2304, out_features=9216, bias=False)
          (down_proj): Linear(in_features=9216, out_features=2304, bias=False)
          (act_fn): GELUTanh()
        )
        (input_layernorm): Gemma2RMSNorm((2304,), eps=1e-06)
        (post_attention_layernorm): Gemma2RMSNorm((2304,), eps=1e-06)
        (pre_feedforward_layernorm): Gemm

In [3]:
from math import sqrt

from transformers_sae.sae import SAE, make_sae_config
from transformers_sae.training import TrainingConfig, TrainingMethod

TRAINING_CACHE_DIR = None
VALIDATION_CACHE_DIR = None
NUM_TRAINING_TOKENS = int(1e4)
EVAL_INTERVAL = int(1e6)
NUM_VALIDATION_TOKENS = int(1e4)
# to match Gemma Scope
D_SAE = 16384
D_MODEL = model.d_model
BASE_SAE_PARAMETERS = 2 * D_SAE * D_MODEL + D_SAE + D_MODEL
D_SAE_INTERACTION = int(
    (
        sqrt(((2 * D_MODEL + 1) ** 2 + 4 * (BASE_SAE_PARAMETERS - D_MODEL)))
        - (2 * D_MODEL + 1)
    )
    / 2
)
INTERACTION_SAE_PARAMETERS = (
    D_SAE_INTERACTION**2 + 2 * D_SAE_INTERACTION * D_MODEL + D_SAE_INTERACTION + D_MODEL
)
TOPK = 100
TOKENIZER_BATCH_SIZE = 256
FINETUNE_FRACTION = 0.1

empty_saes = {
    layer: SAE(
        make_sae_config(
            d_model=model.d_model,
            d_sae=D_SAE_INTERACTION,
            device=TRAINING_DEVICE,
            train_dtype=torch.float32,
            inference_dtype=torch.bfloat16,
            encoder_kind="batch_topk",
            top_k=TOPK,
            with_interaction=True,
        )
    )
    for layer in range(model.num_layers)
}


def linear_decay_during_finetune(frac_trained: float):
    if frac_trained < (1 - FINETUNE_FRACTION):
        return 1.0
    return 1.0 - (frac_trained - (1 - FINETUNE_FRACTION)) / FINETUNE_FRACTION


training_config = TrainingConfig(
    tokenizer_batch_size=TOKENIZER_BATCH_SIZE,
    training_batch_size=TRAINING_BATCH_SIZE,
    num_train_tokens=NUM_TRAINING_TOKENS,
    eval_interval=EVAL_INTERVAL,
    train_layers=list(range(0, model.num_layers)),
    betas=(
        0.0,
        0.999,
    ),  # TODO: is this actually good for our training method? not for tinystories anyway
    lr=1e-4,
    interaction_lr=1e-4,
    lr_schedule=linear_decay_during_finetune,  # per Karvonen (2025)
    downstream_reconstruction_weight=1.0,
    reconstruction_weight=1.0,
    balance_reconstruction_losses=True,
    method=TrainingMethod.next_layer,
)

training_results = {}
validation_results = {}

In [ ]:
from transformers_sae.training import train

with (
    torch._subclasses.fake_tensor.FakeTensorMode(
        allow_non_fake_inputs=True,
        shape_env=torch.fx.experimental.symbolic_shapes.ShapeEnv(
            should_record_events=False
        ),
    ),
    MemoryTrackingMode() as mm,
):
    training_results = train(
        model,
        tokenizer,
        empty_saes,
        training_dataset,
        training_config,
    )

print(mm.memory_max, mm.memory_cur)

{'meta': '7.31 GB', 'mps:0': '21.30 GB', 'cpu': '58.76 MB'} {'meta': '0 B', 'mps:0': '5.32 GB', 'cpu': '0 B'}
